# Data Engineering et Machine Learning avec Snowflake
## Prédiction du Prix des Maisons

## Etape 0 - Mise en place & configuration de l'environnement
Avant de commencer, on configure la base de données, le warehouse et le contexte de travail.

In [ ]:
-- Création de la base
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;

-- Création du warehouse
CREATE WAREHOUSE IF NOT EXISTS HOUSE_PRICE_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 120
  AUTO_RESUME = TRUE;

-- Définition du contexte
USE DATABASE HOUSE_PRICE_DB;
USE SCHEMA PUBLIC;
USE WAREHOUSE HOUSE_PRICE_WH;

In [ ]:
import pandas as pd
import seaborn as sns
from scipy.stats import f_oneway
import numpy as np
import sklearn
import json
import xgboost
import streamlit as st
from snowflake.ml.registry import Registry
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Étape 1 — Ingestion des données

### 1.1 Création du Stage S3

On crée un stage Snowflake pointant vers le bucket S3 sans définir de format a priori,
car on ne connaît pas encore le type de fichier.

In [ ]:
CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
    URL = 's3://logbrain-datalake/datasets/house_price/'
    FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE);

-- Vérification : liste les fichiers du stage
LIST @HOUSE_PRICE_STAGE;

### Interprétation 

La connexion au bucket S3 est établie. On identifie un seul fichier : 
**Housing_Price_Data.json**.

L'extension `.json` confirme le format des données. 
On va donc charger ce fichier en tant que JSON en utilisant le format VARIANT.

### 1.2 Inspection du contenu

In [ ]:
-- Table de staging pour stocker le JSON brut
CREATE OR REPLACE TABLE HOUSE_PRICE_RAW (raw VARIANT);

-- Chargement du fichier depuis le stage existant
COPY INTO HOUSE_PRICE_RAW
    FROM @HOUSE_PRICE_STAGE/Housing_Price_Data.json
    FILE_FORMAT = (TYPE = JSON)
    ON_ERROR = 'CONTINUE';

In [ ]:
session = get_active_session()

df = session.table("HOUSE_PRICE_RAW")
count = df.count()

if count > 0:
    print(f"Chargement réussi — {count} ligne(s) chargée(s) dans HOUSE_PRICE_RAW")
else:
    print("La table est vide, vérifier le COPY INTO")

### Interprétation

Le chargement s'est bien passé. Le statut est LOADED avec 1 ligne parsée et
1 ligne chargée : le fichier JSON est traité comme un seul objet, ce qui est normal
pour un JSON array. 0 erreur détectée.


### 1.3 Inspection de la structure JSON

In [ ]:
session = get_active_session()
row = session.table("HOUSE_PRICE_RAW").limit(1).collect()[0]
data = json.loads(row[0])

print("Colonnes détectées :")
for col, val in data[0].items():
    print(f"  - {col} : {type(val).__name__} (ex: {val})")

### Interprétation

On identifie 13 colonnes, toutes stockées en string dans le JSON, y compris
les colonnes numériques comme price, area, bedrooms, etc.
On distingue deux types de colonnes :
- Numériques à caster en NUMBER : price, area, bedrooms, bathrooms, stories, parking
- Catégorielles à garder en VARCHAR : airconditioning, basement, furnishingstatus,
  guestroom, hotwaterheating, mainroad, prefarea

### 1.4 Aplatissement du JSON (Flattening)

Maintenant qu'on connaît la structure exacte du fichier, on utilise LATERAL FLATTEN
pour extraire chaque enregistrement et le transformer en colonnes structurées
avec les bons types.

In [ ]:
CREATE OR REPLACE TABLE HOUSE_PRICE AS
SELECT
    value:price::NUMBER             AS PRICE,
    value:area::NUMBER              AS AREA,
    value:bedrooms::NUMBER          AS BEDROOMS,
    value:bathrooms::NUMBER         AS BATHROOMS,
    value:stories::NUMBER           AS STORIES,
    value:parking::NUMBER           AS PARKING,
    value:mainroad::VARCHAR         AS MAINROAD,
    value:guestroom::VARCHAR        AS GUESTROOM,
    value:basement::VARCHAR         AS BASEMENT,
    value:hotwaterheating::VARCHAR  AS HOTWATERHEATING,
    value:airconditioning::VARCHAR  AS AIRCONDITIONING,
    value:prefarea::VARCHAR         AS PREFAREA,
    value:furnishingstatus::VARCHAR AS FURNISHINGSTATUS
FROM HOUSE_PRICE_RAW,
LATERAL FLATTEN(input => raw);


## Étape 2 — Chargement et conversion en Pandas

In [ ]:
session = get_active_session()

# Chargement depuis Snowflake
df = session.table("HOUSE_PRICE")
df.show(10)

# Conversion en Pandas
df_house_price = df.to_pandas()
print(f"Conversion réussie — {df_house_price.shape[0]} lignes, {df_house_price.shape[1]} colonnes")

### Interprétation

Les données sont bien chargées depuis Snowflake et converties en Pandas DataFrame.
On retrouve nos 1090 lignes et 13 colonnes attendues. On peut maintenant utiliser
toute la puissance de Pandas pour l'exploration et le traitement des données.

## Etapes 3 - Exploration des données

L'exploration des données est une étape essentielle avant de construire un modèle ML.
Elle nous permet de comprendre la structure du dataset, détecter les valeurs manquantes,
les outliers et comprendre les relations entre les variables.

### 3.1 Exploration des données quantitative

### 3.1.1- Statistiques descriptives

In [ ]:
df_house_price.describe()

### Interprétation

Aucune valeur manquante détectée sur l'ensemble des 13 colonnes.
Le dataset est complet.

**PRICE :**
- Moyenne (237 663) > médiane (213 500) et écart-type élevé (94 090).
- Cela suggère une possible asymétrie à droite avec des prix élevés 
  qui tirent la moyenne vers le haut.

**AREA :**
- Moyenne (102) > médiane (92) avec un écart-type de 42.
- Même tendance que PRICE, possible asymétrie à droite.

**BEDROOMS / BATHROOMS / STORIES :**
- Distributions concentrées sur de petites valeurs (médiane = 3, 1, 2).
- Possiblement des distributions discrètes avec quelques valeurs extrêmes.

**PARKING :**
- 75% des maisons ont entre 0 et 1 place, max à 3.
- Distribution probablement concentrée sur 0 et 1.

Ces observations restent des suppositions basées sur les statistiques.
La moyenne et la médiane seules ne suffisent pas à confirmer la forme 
de la distribution.

On va visualiser les histogrammes pour confirmer ces hypothèses 
et choisir la méthode de détection des outliers la plus adaptée.

### 3.1.2 Distribution des variables numériques

In [ ]:

df_house_price.hist(bins=30, figsize=(10, 5), layout=(2, 3))
plt.show()

### Interprétation

Les distributions confirment notre hypothèse d'asymétrie à droite 
pour **PRICE** et **AREA**.

Les variables **BEDROOMS**, **BATHROOMS**, **STORIES** et **PARKING** 
sont des variables discrètes concentrées sur de petites valeurs, 
leurs valeurs extrêmes seront analysées avec précaution.

La méthode Boxplot et **IQR** sont les plus adaptées pour détecter les outliers 
sur ce dataset. Prochaine étape : détection des outliers.

## 3.1.3- Détection des outliers

On utilise une approche combinée :
1. **Boxplot** pour visualiser les outliers
2. **IQR** pour les quantifier précisément

### 3.1.3.1- Visualisation des outliers(Boxplot)

In [ ]:
cols = df_house_price.select_dtypes(include=['number']).columns

# On crée une figure adaptée au nombre de colonnes
plt.figure(figsize=(15, 8))

for i, col in enumerate(cols):
    plt.subplot(2, 3, i + 1) # On utilise le layout 2x3
    sns.boxplot(y=df_house_price[col])
    plt.title(col)

plt.tight_layout()

### Interprétation

Les boxplots confirment visuellement la présence d'outliers :

**PRICE et AREA :**
- Plusieurs points au-dessus de la moustache supérieure, confirmant 
  l'asymétrie à droite observée dans les histogrammes.
- Les outliers sont des prix et surfaces très élevés par rapport au reste.

**BEDROOMS :**
- Quelques outliers à 5 et 6 chambres, des maisons atypiques 
  mais qui peuvent exister dans la réalité.

**BATHROOMS, STORIES, PARKING :**
- Un seul point outlier visible pour chaque variable (valeur maximale).
- Ces valeurs extrêmes sont marginales mais réelles.

**Conclusion :**
- Les outliers sont principalement concentrés sur **PRICE** et **AREA**.
- Pour les variables discrètes, les valeurs extrêmes semblent 
  être des cas rares mais plausibles.

Nous allons donc les analyser(**PRICE et AREA**) de plus près avec la methode IQR pour vérifier leur cohérence metier par rapport 
aux autres variables.

### 3.1.3.2.- Isolation des outliers PRICE et AREA(methode IQR)

In [ ]:
# Calcul des bornes IQR pour PRICE et AREA
for column in ['PRICE', 'AREA']:
    Q1 = df_house_price[column].quantile(0.25)
    Q3 = df_house_price[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df_house_price[df_house_price[column] > upper]
    print(f"\n--- Outliers {column} (> {upper:.0f}) : {len(outliers)} lignes ---")
    print(outliers.to_string())

### Interprétation 

Les outliers observés sur PRICE et AREA semblent cohérents d'un point de vue 
métier. On choisit de les conserver.

PRICE étant notre variable cible avec une distribution asymétrique à droite, 
nous aurons à appliquer une transformation logarithmique lors de la préparation 
des données pour stabiliser sa distribution et limiter l'impact des valeurs 
extrêmes sur les modèles.

### 3.1.4 Analyse de corrélation

La matrice de corrélation nous permet de comprendre les relations entre
les variables numériques et la variable cible PRICE.

In [ ]:
import seaborn as sns

numeric_cols = df_house_price.select_dtypes(include=['number']).columns

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(df_house_price[numeric_cols].corr(), annot=True, fmt=".2f", 
            cmap="coolwarm", linewidths=0.5)
plt.title("Matrice de corrélation")
plt.show()

### Interprétation

La matrice confirme ce qu'on pouvait intuitivement supposer : plus une maison 
est grande et bien équipée, plus elle est chère. Les variables explicatives 
n'interfèrent pas entre elles, ce qui nous donne une bonne base pour 
construire nos modèles.

## 3.2 Distribution des variables catégorielles

## 3.2.1- Identification des modalités rares

Avant l'analyse de corrélation, on vérifie la distribution des modalités 
des variables catégorielles pour identifier celles qui sont sous-représentées. 
Une modalité rare peut biaiser le modèle ou poser des problèmes lors du split train/test.

In [ ]:
categorical_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 
                    'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']

for column in categorical_cols:
    plt.figure(figsize=(5, 3)) 
    
    repartition = df_house_price[column].value_counts().index
    sns.countplot(data=df_house_price, x=column, hue=column, palette="viridis", 
                  order=repartition, legend=False)
    
    proportion = df_house_price[column].value_counts(normalize=True)
    
    for i, prop in enumerate(proportion):
        plt.text(i, prop * len(df_house_price), f"{prop: .2%}", ha="center", va="bottom")
    
    plt.title(f"Distribution de la variable {column}")
    plt.ylabel("Nombre de valeurs")
    plt.xlabel(column)
    plt.show()

### Interprétation

La majorité des variables catégorielles présentent une distribution acceptable 
entre leurs modalités.

On attire cependant l'attention sur **HOTWATERHEATING** dont la modalité "yes" 
ne représente que **4.77%** des observations. Dans le marché immobilier de ce 
dataset, le chauffage à eau chaude est quasi absent, ce qui rend cette variable 
peu discriminante pour expliquer les variations de prix. Elle sera donc écartée 
lors de la préparation des données.

## 4 - Préparation des données

### 4.1- Suppression de HOTWATERHEATING

In [ ]:
df_model = df_house_price.drop(columns=['HOTWATERHEATING'])
print(f"Colonnes restantes : {df_model.columns.tolist()}")

### 4.2- Transformation logarithmique de PRICE

In [ ]:
df_model['PRICE_LOG'] = np.log(df_model['PRICE'])
df_model = df_model.drop(columns=['PRICE'])
print("Transformation log appliquée sur PRICE")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(df_model['PRICE_LOG'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title("Boxplot de PRICE_LOG")
axes[0].set_ylabel("Valeur")

# Distribution avec KDE
sns.histplot(df_model['PRICE_LOG'], bins=30, kde=True, 
             color='steelblue', ax=axes[1])
axes[1].set_title("Distribution de PRICE_LOG")
axes[1].set_xlabel("Valeur")
axes[1].set_ylabel("Fréquence")

plt.suptitle("Analyse de PRICE après transformation log", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Interprétation

La transformation logarithmique a bien réduit l'asymétrie de PRICE. La distribution
se rapproche d'une forme plus symétrique et la courbe KDE confirme cette normalisation.
Le boxplot ne montre plus qu'un seul outlier résiduel, contre plusieurs avant la
transformation ce qui valide notre choix d'appliquer le log.

### 4.3- Sélection des variables catégorielles - ANOVA

In [ ]:
from scipy.stats import f_oneway
import matplotlib.pyplot as plt
import pandas as pd

categorical_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 
                    'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']

results = []
for col in categorical_cols:
    groups = [df_model[df_model[col] == mod]['PRICE_LOG'].values 
              for mod in df_model[col].unique()]
    f_stat, p_value = f_oneway(*groups)
    results.append({'Variable': col, 'F-stat': f_stat})

df_anova = pd.DataFrame(results).sort_values('F-stat', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(df_anova['Variable'], df_anova['F-stat'], color='steelblue')
plt.title("F-statistics des tests ANOVA")
plt.xlabel("F-statistique")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

categorical_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 
                    'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    return np.sqrt(phi2 / min(r-1, k-1))

# Calcul de la matrice de V de Cramer
matrix = pd.DataFrame(index=categorical_cols, columns=categorical_cols)

for col1 in categorical_cols:
    for col2 in categorical_cols:
        matrix.loc[col1, col2] = cramers_v(df_model[col1], df_model[col2])

matrix = matrix.astype(float)

# Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Matrice de V de Cramer — Association entre variables catégorielles")
plt.tight_layout()
plt.show()

### Interprétation

La matrice de V de Cramer confirme qu'aucune paire de variables catégorielles 
ne présente une association forte. Toutes les valeurs hors diagonale sont 
inférieures à 0.5, avec un maximum de 0.37 entre GUESTROOM et BASEMENT.

Cette légère association entre GUESTROOM et BASEMENT est métier cohérente — 
les maisons disposant d'une chambre d'amis ont tendance à avoir également 
un sous-sol, ce qui reflète des biens de plus grande taille. Elle reste 
cependant trop faible pour justifier l'élimination d'une des deux variables.

Toutes les variables catégorielles sont indépendantes entre elles et 
seront conservées pour l'entraînement des modèles.

Prochaine étape : encodage des variables catégorielles.

### 4.4- Encodage des variables catégorielles

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df_model.copy()

# Label Encoding pour les variables binaires
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'AIRCONDITIONING', 'PREFAREA']
le = LabelEncoder()
for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# One-Hot Encoding pour FURNISHINGSTATUS
df_encoded = pd.get_dummies(df_encoded, columns=['FURNISHINGSTATUS'], drop_first=True)

print(f"Encodage terminé — {df_encoded.shape[1]} colonnes")
df_encoded.head()

### Interprétation

L'encodage s'est bien passé. On obtient 13 colonnes au total.

Les variables binaires sont encodées en 0/1. FURNISHINGSTATUS génère 
2 colonnes avec "furnished" comme modalité de référence — ce choix est 
adapté à la régression linéaire qu'on va utiliser en premier modèle 
pour éviter la multicolinéarité.

Prochaine étape : séparation des features (X) et de la variable cible (y), 
puis split train/test.

## 5.1 — Séparation X/y et split train/test

On sépare les features (X) de la variable cible (y), puis on divise 
le dataset etant petit (1090) en un ensemble d'entraînement (70%) et un ensemble de test (30%) pour avoir assez de donnees de test.

In [ ]:
from sklearn.model_selection import train_test_split

# Séparation X/y
X = df_encoded.drop(columns=['PRICE_LOG'])
y = df_encoded['PRICE_LOG']

# Split train/test 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

## 5.2 — Normalisation des données

La régression linéaire est sensible à l'échelle des variables. 
On applique une normalisation (StandardScaler) pour centrer et réduire 
les features à une moyenne de 0 et un écart-type de 1.


On fit le scaler uniquement sur le train et on le transforme 
sur le train et le test pour éviter tout data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Normalisation appliquée")
print(f"  Moyenne X_train : {X_train_scaled.mean().round(2)}")
print(f"  Écart-type X_train : {X_train_scaled.std().round(2)}")

### Interprétation

La normalisation s'est bien appliquée :
- Moyenne ~= 0 (le -0.0 est un artefact d'arrondi)
- Écart-type = 1.0

Toutes les features sont maintenant sur la même échelle, 
ce qui est indispensable pour la régression linéaire.

Prochaine étape : entraînement du premier modèle: Régression Linéaire.

# 6- Entraînement des modèles

## 6.1 Régression Linéaire (Baseline)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

print("Modèle entraîné")
print(f"\nCoefficients :")
for col, coef in zip(X_train.columns, lr.coef_):
    print(f"  - {col} : {coef:.4f}")
print(f"\nIntercept : {lr.intercept_:.4f}")

### Interprétation

Le modèle a bien été entraîné. En analysant les coefficients :

**Variables les plus influentes :**
- **AREA (0.1047)** : confirme ce qu'on avait observé dans la matrice 
  de corrélation — la surface est le facteur le plus déterminant du prix.
- **BATHROOMS (0.0909)** : le nombre de salles de bain arrive en deuxième 
  position, cohérent avec notre analyse.
- **AIRCONDITIONING (0.0668)** : la climatisation a un impact positif 
  significatif sur le prix.

**Point notable :**
- **FURNISHINGSTATUS_unfurnished (-0.0512)** est le seul coefficient négatif, 
  ce qui est cohérent — une maison non meublée vaut moins qu'une maison meublée.

**Intercept (12.3132)** : représente la valeur de base de PRICE_LOG 
quand toutes les features sont à 0.

Prochaine étape : Prédiction du modèle.

In [ ]:
# Prédictions sur le jeu de test
y_pred = lr.predict(X_test_scaled)

# Calcul des résidus
residuals = y_test - y_pred

print(f"Prédictions effectuées — {len(y_pred)} observations")
print(f"Moyenne des résidus : {residuals.mean():.4f}")

### Interprétation

Les prédictions ont été effectuées sur les 327 observations du jeu de test.
La moyenne des résidus est quasi nulle (-0.0250 ≈ 0), ce qui est un bon 
premier indicateur — le modèle ne surestime ni ne sous-estime systématiquement.

On evalue maintenant la performance du modele

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Inversion de la transformation log
y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred)

r2   = r2_score(y_test, y_pred)  # R² reste en log car c'est une proportion
mae  = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print("Performances — Régression Linéaire")
print("-" * 40)
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:.0f} €")
print(f"  RMSE : {rmse:.0f} €")

### Interprétation

Le modèle de Régression Linéaire, entraîné sur les logarithmes des prix pour stabiliser la variance, affiche un Rcarre de 0,6764.

- Ce que cela signifie : Le modèle capture environ 68% de l'information. C'est une base de travail correcte, mais insuffisante pour une mise en production fiable. Il nous manque encore 32% de la logique du marché immobilier (effets de quartier, combinaisons de critères, etc.).

- L'erreur concrète : La MAE (Erreur Moyenne Absolue) est de 38 692 €. En clair, pour une maison moyenne, notre estimation se trompe de près de 39 000 €. C'est une marge d'erreur trop élevée pour un acheteur ou un vendeur.

- Le signal d'alerte : La RMSE (51 137 €) est significativement plus haute que la MAE. Cela prouve que le modèle fait de grosses erreurs de prédiction sur certaines propriétés (probablement les maisons les plus chères ou les plus atypiques).

La limite ici n'est pas la qualité des données, mais la rigidité mathématique du modèle. La régression linéaire part du principe que chaque critère (surface, chambres) s'ajoute de manière constante. Or, l'immobilier est complexe :

-- Effets de seuil : Une 3ème chambre apporte souvent plus de valeur qu'une 5ème.
-- Interactions : Une piscine a beaucoup plus de valeur dans une villa de luxe que dans un petit pavillon de banlieue. 
La ligne droite ne voit pas ces nuances.

Pour franchir cette limite, nous allons entrainer deux autres modeles qui sont beaucoup plus robustes: **Random Forest et XGBoost.**

## 6.2.- Entraînement Random Forest et XGBoos

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 1. Entraînement du Random Forest
# On utilise 100 arbres pour commencer, c'est une valeur sûre
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# 2. Entraînement du XGBoost
# Ce modèle est souvent plus rapide et plus précis
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.3, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

print("Entraînement terminé pour les deux modèles !")

### 6.2.1- Prédictions, évaluation et comparaison des modèles

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# 1. Prédictions (en Log)
y_pred_rf_log = rf_model.predict(X_test_scaled)
y_pred_xgb_log = xgb_model.predict(X_test_scaled)

# 2. Conversion en Euros (€)
y_test_euro = np.exp(y_test)
y_pred_rf_euro = np.exp(y_pred_rf_log)
y_pred_xgb_euro = np.exp(y_pred_xgb_log)

# 3. Calcul des scores pour chaque modèle
results = {
    'Modèle': ['Régression Linéaire', 'Random Forest', 'XGBoost'],
    'R²': [
        r2_score(y_test, y_pred), # Ton score précédent (0.6764)
        r2_score(y_test, y_pred_rf_log),
        r2_score(y_test, y_pred_xgb_log)
    ],
    'MAE (€)': [
        38692, # Ta MAE précédente
        mean_absolute_error(y_test_euro, y_pred_rf_euro),
        mean_absolute_error(y_test_euro, y_pred_xgb_euro)
    ]
}

# Affichage propre sous forme de tableau
df_results = pd.DataFrame(results)
print("Comparaison des Modèles — Prix des Maisons")
print("-" * 50)
print(df_results.to_string(index=False))

### Bilan de la comparaison des modèles

La régression linéaire montre rapidement ses limites avec une MAE de 38 692 € — 
un modèle trop rigide pour capturer la complexité du marché immobilier.

Les modèles basés sur les arbres apportent une amélioration significative. 
Le Random Forest divise l'erreur par deux (19 817 €), mais c'est le XGBoost 
qui s'impose avec la meilleure MAE (15 852 €) grâce à sa capacité à apprendre 
de ses propres erreurs de manière séquentielle.

Cependant, le RMSE encore élevé (31 880 €) révèle que le modèle fait 
des écarts importants sur certaines maisons atypiques. C'est pourquoi 
on choisit d'optimiser uniquement le XGBoost — le meilleur candidat — 
pour tenter de réduire ces erreurs extrêmes et franchir 
la barre des 15 000 € de MAE.

Prochaine étape : optimisation des hyperparamètres du XGBoost.

## Optimisation des hyperparamètres du XGBoost.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Configuration et Entraînement éclair
xgb_final = XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    early_stopping_rounds=20,
    random_state=42
)

xgb_final.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)], 
    verbose=False 
)

# 2. Prédictions et conversion (Log -> Euro)
y_pred_log = xgb_final.predict(X_test_scaled)
y_pred_euro = np.exp(y_pred_log)
y_test_euro = np.exp(y_test)

# 3. Calcul de toutes les métriques
mae_final  = mean_absolute_error(y_test_euro, y_pred_euro)
rmse_final = np.sqrt(mean_squared_error(y_test_euro, y_pred_euro))
r2_final   = r2_score(y_test, y_pred_log)

print("BILAN FINAL : XGBOOST OPTIMISÉ")
print("-" * 40)
print(f"  Nombre d'arbres utilisés : {xgb_final.best_iteration}")
print(f"  R² (Précision globale)   : {r2_final:.4f}")
print(f"  MAE (Erreur moyenne)     : {mae_final:,.0f} €")
print(f"  RMSE (Impact outliers)   : {rmse_final:,.0f} €")

In [ ]:
import streamlit as st
import pandas as pd

results_data = {
    'Modèle': ['Régression Linéaire', 'Random Forest', 'XGBoost', 'XGBoost Optimisé'],
    'R²': [0.6764, 0.8500, 0.8394, 0.8620],
    'MAE (€)': [38692, 19817, 15852, 14402],
    'RMSE (€)': [51137, 32210, 31880, 28474]
}

results_df = pd.DataFrame(results_data)

st.write("Comparaison des modèles :")
st.dataframe(results_df.style.highlight_max(subset=['R²'], color='green')
                              .highlight_min(subset=['MAE (€)', 'RMSE (€)'], color='green'))

st.subheader("Meilleur modèle : XGBoost Optimisé")
st.write(f"R² : `{0.8620:.4f}`")
st.write(f"MAE : `14402 €`")
st.write(f"RMSE : `28474 €`")

st.write("Paramètres du meilleur modèle :")
st.write(f"- n_estimators : `{xgb_final.n_estimators}`")
st.write(f"- learning_rate : `{xgb_final.learning_rate}`")
st.write(f"- max_depth : `{xgb_final.max_depth}`")
st.write(f"- early_stopping_rounds : `{xgb_final.early_stopping_rounds}`")

### Interprétation

Le XGBoost optimisé est notre meilleur modèle avec un R² de 0.86 
et une MAE de 14 402 €, soit une réduction de 63% de l'erreur 
par rapport à notre baseline (régression linéaire à 38 692 €).

Le modèle capture 86% des mécaniques de prix du marché immobilier. 
Les 14% restants s'expliquent probablement par des critères subjectifs 
difficilement quantifiables comme l'état de finition intérieure 
ou l'attrait du bien.

Le fait que l'early stopping se soit déclenché à 300 arbres confirme 
que le modèle a trouvé le bon équilibre entre précision et généralisation, 
sans tomber dans le surapprentissage.

Ce modèle est notre candidat pour le Model Registry.

## 7- Stockage du modèle dans le Snowflake Model Registry

Le Model Registry de Snowflake permet de versionner, tracer et gérer 
les modèles ML directement dans Snowflake. On y enregistre notre 
meilleur modèle — le XGBoost optimisé — avec ses métadonnées.

In [ ]:
from snowflake.ml.registry import Registry
from datetime import datetime
import warnings

# Suppression des warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

registry = Registry(session=session)

# Nommage avec timestamp
model_name = "HOUSE_PRICE_XGBOOST"
model_version = f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Nettoyage des colonnes
X_train_clean = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_train_clean.columns = X_train_clean.columns.str.replace('-', '_')

model_ref = registry.log_model(
    model=xgb_final,
    model_name=model_name,
    version_name=model_version,
    sample_input_data=X_train_clean.head(5),
    metrics={
        'r2': float(0.8620),
        'mae': float(14402),
        'rmse': float(28474)
    },
    options={
        "case_sensitive": True,
    },
    comment="XGBoost optimisé — meilleur modèle de prédiction de prix immobilier"
)

warnings.resetwarnings()

st.write("Modèle enregistré avec succès !")
st.write(f"Model Name : `{model_name}`")
st.write(f"Version : `{model_version}`")

## 8 - Inférence avec le modèle enregistré

On charge le modèle depuis le Model Registry et on l'utilise 
pour générer des prédictions sur de nouvelles données.

In [ ]:
# Chargement du modèle depuis le registry
model = registry.get_model("HOUSE_PRICE_XGBOOST").version(model_version)

print(f"Modèle {model_version} chargé depuis le Snowflake Model Registry")

In [ ]:
# Préparation des nouvelles données
X_new = pd.DataFrame(X_test_scaled, columns=X_train.columns)
X_new.columns = X_new.columns.str.replace('-', '_')

# Prédictions
y_pred_log = model.run(X_new, function_name="predict")

# Inversion de la transformation log
y_pred_euros = np.exp(y_pred_log.values.flatten())
y_test_euros = np.exp(y_test.values)

# Affichage des résultats
results_inference = pd.DataFrame({
    'Prix Réel (€)': y_test_euros.astype(int),
    'Prix Prédit (€)': y_pred_euros.astype(int),
    'Écart (€)': (y_pred_euros - y_test_euros).astype(int)
})

st.write("Résultats de l'inférence :")
st.dataframe(results_inference.head(10))

### Interprétation

L'inférence fonctionne correctement. Sur les 10 premières prédictions :

- La majorité des écarts sont faibles et cohérents avec notre MAE de 14 402 €.
- On note cependant la ligne 6 avec un écart de 79 996 €, une maison 
  atypique que le modèle a du mal à estimer, ce qui explique pourquoi 
  notre RMSE (28 474 €) reste supérieur à la MAE.
- Les lignes 0 et 1 ont exactement le même prix réel et prédit, 
  ce qui suggère des maisons avec des caractéristiques identiques 
  dans notre dataset.

Le modèle est opérationnel et prêt à être utilisé dans une application métier.

Prochaine et dernière étape : construction de l'application Streamlit.

## 9 - Application Streamlit

On développe une application Streamlit directement dans Snowflake 
pour permettre aux utilisateurs métier d'estimer le prix d'une maison 
en saisissant ses caractéristiques et en obtenant une prédiction en temps réel.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import json
import time
from snowflake.snowpark.context import get_active_session

st.cache_data.clear()  # Vider le cache

session = get_active_session()

@st.cache_data
def load_scaler_stats():
    df = session.table("HOUSE_PRICE_DB.PUBLIC.SCALER_STATS").to_pandas()
    means = dict(zip(df['FEATURE'].str.upper(), df['MEAN']))
    stds  = dict(zip(df['FEATURE'].str.upper(), df['STD']))
    return means, stds

means, stds = load_scaler_stats()

st.title("🏠 Estimation du Prix d'une Maison")
st.write("Renseignez les caractéristiques de la maison pour obtenir une estimation du prix.")

col1, col2, col3 = st.columns(3)

with col1:
    area      = st.number_input("Surface (m²)", min_value=33, max_value=324, value=100)
    bedrooms  = st.slider("Chambres", 1, 6, 3)
    bathrooms = st.slider("Salles de bain", 1, 4, 1)
    stories   = st.slider("Étages", 1, 4, 2)

with col2:
    parking         = st.slider("Places de parking", 0, 3, 1)
    mainroad        = st.selectbox("Route principale", ["yes", "no"])
    guestroom       = st.selectbox("Chambre d'amis", ["yes", "no"])
    basement        = st.selectbox("Sous-sol", ["yes", "no"])

with col3:
    airconditioning  = st.selectbox("Climatisation", ["yes", "no"])
    prefarea         = st.selectbox("Zone privilégiée", ["yes", "no"])
    furnishingstatus = st.selectbox("Ameublement", 
                                    ["furnished", "semi-furnished", "unfurnished"])

if st.button("Estimer le prix", type="primary"):
    start_time = time.time()
    
    with st.spinner("Calcul en cours..."):
        
        # Encodage
        input_data = {
    'AREA'                               : area,
    'BEDROOMS'                           : bedrooms,
    'BATHROOMS'                          : bathrooms,
    'STORIES'                            : stories,
    'PARKING'                            : parking,
    'MAINROAD'                           : 1 if mainroad == "yes" else 0,
    'GUESTROOM'                          : 1 if guestroom == "yes" else 0,
    'BASEMENT'                           : 1 if basement == "yes" else 0,
    'AIRCONDITIONING'                    : 1 if airconditioning == "yes" else 0,
    'PREFAREA'                           : 1 if prefarea == "yes" else 0,
    'FURNISHINGSTATUS_SEMI_FURNISHED'    : 1 if furnishingstatus == "semi-furnished" else 0,
    'FURNISHINGSTATUS_UNFURNISHED'       : 1 if furnishingstatus == "unfurnished" else 0,
}

        # Normalisation manuelle sans sklearn
        input_scaled = {col: (val - means[col]) / stds[col] 
                        for col, val in input_data.items()}
        
        input_df = pd.DataFrame([input_scaled])

        try:
            # Prédiction via SQL
            input_snowpark = session.create_dataframe(input_df)
            input_snowpark.create_or_replace_temp_view("TEMP_HOUSE_PREDICTION")

            result = session.sql("""
                SELECT HOUSE_PRICE_XGBOOST!PREDICT(*) AS predicted_price 
                FROM TEMP_HOUSE_PREDICTION
            """).collect()

            pred_log   = json.loads(result[0]['PREDICTED_PRICE'])
            pred_value = float(list(pred_log.values())[0])
            pred_euros = int(np.exp(pred_value))

            end_time = time.time()

            st.success(f"💰Prix estimé : **{pred_euros:,} €**")
            st.caption(f"⏱️ Prédiction effectuée en {(end_time - start_time):.2f} secondes")

        except Exception as e:
            st.error(f"❌ Erreur : {str(e)}")